# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display high-level metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we print out the record sets (tables), and for each, the available fields and columns identified by their `@id`.

In [ ]:
# List all record sets and their fields/columns by their @id
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in the schema.")
else:
    for recset in record_sets:
        print(f"RecordSet: {recset['@id']}")
        fields = recset.get('field', [])
        print(f"  Fields: {[fld['@id'] for fld in fields]}")
        if 'column' in recset:
            print(f"  Columns: {[col['@id'] for col in recset['column']]}")
        print('-' * 60)

If your record sets list is empty, it's possible the metadata uses `distribution` objects for tabular data. Let's try printing some sample records from a typical main record set by inspecting the schema. For the FAIR² dataset, the main record set ID is likely `'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'` (as seen in the distribution list). Let's inspect a sample:

In [ ]:
# For FAIR², the main record set is typically at the distribution @id, so we use that.
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
try:
    sample_iter = dataset.records(record_set=main_record_set_id)
    for i, record in enumerate(sample_iter):
        print(f"Sample row {i+1}: {record}")
        if i > 2:
            break
except Exception as e:
    print(f"Could not print sample records: {e}")

## 3. Data Extraction
Load data from record sets into a DataFrame for analysis. All reading and manipulation is referenced by the record set and field `@id`s.

In [ ]:
# List your record set IDs here. Typically you can extract this programmatically, but for this dataset, we use the known distribution @id.
record_set_ids = ['https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034']
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns of '{record_set_id}':")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing and filtering. The following samples select a numeric field for filtering and normalization, and a field for grouping. Adjust the IDs to match the available columns printed above.

In [ ]:
# Choose a numeric field (e.g., 'MW[kDa]' or 'Coverage[%]') and a group field (e.g., 'Description'):
# Please replace field IDs with actual column names as identified in the previous output.
df = dataframes[record_set_ids[0]]
print("Columns available for analysis:", df.columns.tolist())

# Typical fields present in this type of proteomics dataset could be: 'MW[kDa]', 'PI', 'Coverage[%]', 'AbundanceSample_1', 'AbundanceSample_2', etc.
# For demonstration, we'll search for a likely present numeric field
numeric_candidates = [c for c in df.columns if any(sub in c.lower() for sub in ["mw", "abundance", "coverage", "count", "score", "intensity"])]

if not numeric_candidates:
    raise ValueError("No numeric candidate columns found.")
numeric_field = numeric_candidates[0]
# Find a group candidate (usually a categorical column such as 'Description')
group_candidates = [c for c in df.columns if any(sub in c.lower() for sub in ["description", "type", "class", "group", "id"])]
group_field = group_candidates[0] if group_candidates else None

print(f"Numeric field chosen: {numeric_field}")
if group_field:
    print(f"Group field chosen: {group_field}")

# Ensure numeric type for analysis
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as an arbitrary threshold

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (top quartile):")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Below, we visualize the distribution of the selected numeric field and show groupwise means if a group field was found and used.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping was done, show the top means per group
if group_field and group_field in df.columns:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:10]
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_means.values, y=group_means.index)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using `mlcroissant`, explored its metadata, extracted tabular data by referencing Croissant `@id` fields, performed basic numeric filtering and normalization, and visualized the results. This approach enables reproducible FAIR dataset analysis using standards-based schema referencing.